# Drei Binomial-Konfidenzintervalle im Vergleich

Für eine beobachtete Trefferzahl $k$ in einer Binomialstichprobe vom Umfang $n$ werden drei Konfidenzintervalle für den unbekannten Parameter $p$ berechnet:

- Wald-Intervall,
- Wilson-Intervall,
- Clopper-Pearson-Intervall.

Das Notebook erzeugt bewusst keine Grafik. Die letzte Zelle gibt ausschließlich die drei berechneten Intervalle aus.

Das Wald-Intervall wird in seiner klassischen Form nicht auf $[0,1]$ gekürzt; mögliche Schwächen an den Rändern bleiben dadurch sichtbar. Das Clopper-Pearson-Intervall invertiert exakte Binomialtests. Wegen der Diskretheit ist seine tatsächliche Überdeckungswahrscheinlichkeit im Allgemeinen mindestens, aber nicht genau, gleich dem vorgegebenen Konfidenzniveau.


## Eingaben

Nur die folgende Zelle muss verändert werden. Die Beobachtung wird durch die ganzzahlige Trefferzahl $k$ eingegeben; die relative Häufigkeit $h=k/n$ wird daraus berechnet.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
import math
from statistics import NormalDist

from scipy.stats import beta


@dataclass(frozen=True)
class Config:
    # Mathematische Eingaben
    n: int = 80
    k: int = 52
    gamma: float = 0.95

    # Darstellung der Zahlenausgabe
    decimal_places: int = 4


cfg = Config()


## Prüfung und mathematische Berechnung

Eingaben, Prüfung, Berechnung und Ausgabe bleiben getrennt. Alle drei Verfahren verwenden dasselbe Konfidenzniveau $gamma$.


In [2]:
@dataclass(frozen=True)
class Calculation:
    h: float
    z: float
    wald: tuple[float, float]
    wilson: tuple[float, float]
    clopper_pearson: tuple[float, float]


def validate_config(cfg: Config) -> None:
    if not isinstance(cfg.n, int) or isinstance(cfg.n, bool) or cfg.n <= 0:
        raise ValueError("n muss eine positive ganze Zahl sein.")
    if not isinstance(cfg.k, int) or isinstance(cfg.k, bool):
        raise ValueError("k muss eine ganze Zahl sein.")
    if not 0 <= cfg.k <= cfg.n:
        raise ValueError("k muss zwischen 0 und n liegen.")
    if not 0.0 < cfg.gamma < 1.0:
        raise ValueError("gamma muss zwischen 0 und 1 liegen.")
    if (
        not isinstance(cfg.decimal_places, int)
        or isinstance(cfg.decimal_places, bool)
        or not 0 <= cfg.decimal_places <= 12
    ):
        raise ValueError(
            "decimal_places muss eine ganze Zahl zwischen 0 und 12 sein."
        )


def wald_interval(
    *,
    n: int,
    k: int,
    z: float,
) -> tuple[float, float]:
    h = k / n
    half_width = z * math.sqrt(h * (1.0 - h) / n)
    return h - half_width, h + half_width


def wilson_interval(
    *,
    n: int,
    k: int,
    z: float,
) -> tuple[float, float]:
    h = k / n
    denominator = 1.0 + z**2 / n
    center = (h + z**2 / (2.0 * n)) / denominator
    half_width = (
        z
        * math.sqrt(
            h * (1.0 - h) / n + z**2 / (4.0 * n**2)
        )
        / denominator
    )
    return center - half_width, center + half_width


def clopper_pearson_interval(
    *,
    n: int,
    k: int,
    gamma: float,
) -> tuple[float, float]:
    alpha = 1.0 - gamma
    lower = (
        0.0
        if k == 0
        else float(beta.ppf(alpha / 2.0, k, n - k + 1))
    )
    upper = (
        1.0
        if k == n
        else float(beta.ppf(1.0 - alpha / 2.0, k + 1, n - k))
    )
    return lower, upper


def calculate(cfg: Config) -> Calculation:
    validate_config(cfg)

    h = cfg.k / cfg.n
    z = NormalDist().inv_cdf((1.0 + cfg.gamma) / 2.0)

    return Calculation(
        h=h,
        z=z,
        wald=wald_interval(n=cfg.n, k=cfg.k, z=z),
        wilson=wilson_interval(n=cfg.n, k=cfg.k, z=z),
        clopper_pearson=clopper_pearson_interval(
            n=cfg.n,
            k=cfg.k,
            gamma=cfg.gamma,
        ),
    )


## Ausgabe

Die Ausgabe enthält nur die drei Konfidenzintervalle. Die Zahl der Nachkommastellen wird in `Config` festgelegt.


In [3]:
def format_interval(
    interval: tuple[float, float],
    *,
    decimal_places: int,
) -> str:
    lower, upper = interval
    return f"[{lower:.{decimal_places}f}; {upper:.{decimal_places}f}]"


def print_results(cfg: Config, calculation: Calculation) -> None:
    confidence = f"{100.0 * cfg.gamma:g}%"
    width = len("Clopper-Pearson-Konfidenzintervall")

    rows = (
        ("Wald-Konfidenzintervall", calculation.wald),
        ("Wilson-Konfidenzintervall", calculation.wilson),
        (
            "Clopper-Pearson-Konfidenzintervall",
            calculation.clopper_pearson,
        ),
    )

    for name, interval in rows:
        formatted = format_interval(
            interval,
            decimal_places=cfg.decimal_places,
        )
        print(f"{confidence}-{name:<{width}}  {formatted}")


calculation = calculate(cfg)
print_results(cfg, calculation)


95%-Wald-Konfidenzintervall             [0.5455; 0.7545]
95%-Wilson-Konfidenzintervall           [0.5408; 0.7455]
95%-Clopper-Pearson-Konfidenzintervall  [0.5352; 0.7533]
